# Data Preprocess

* Download raw OpTC dataset from https://drive.google.com/drive/folders/1n3kkS3KR31KUegn42yk3-e6JkZvf0Caa, **maintain the original directory structure**.

## 1. OpTC host data

* We directly use the same test data as in the [FLASH](https://github.com/DART-Laboratory/Flash-IDS) paper. 
* We use the same ground-truth labels as in the [FLASH](https://github.com/DART-Laboratory/Flash-IDS) paper. 

In [ ]:
import gzip
import io
import os


def extract_logs(filepath, hostid, output_filename, pattern='wb'):
    """unzip the OpTC dataset, filter out events of the host id

    filepath: the zip file path
    hostid: the host id of this data file
    """
    search_pattern = f'SysClient{hostid}'
    with gzip.open(filepath, 'rt', encoding='utf-8') as fin:
        with open(output_filename, pattern) as f:
            out = io.BufferedWriter(f)
            for line in fin:
                if search_pattern in line:
                    out.write(line.encode('utf-8'))
            out.flush()

    print(f'extract {hostid} in {filepath} to {output_filename}')


raw_optc_path = "G:/datasets/OPTC/" # replaced with your download path
data_map = {
    'optc_day23': {
        'path': 'ecar/benign/20-23Seq19/AIA-201-225/',
        'hostid': '0201',
        'test': ["ecar/evaluation/23Sep19-red/AIA-201-225/AIA-201-225.ecar-2019-12-08T11-05-10.046.json.gz",
                "ecar/evaluation/23Sep19-red/AIA-201-225/AIA-201-225.ecar-last.json.gz"]
    },
    'optc_day24': {
        'path': 'ecar/benign/20-23Seq19/AIA-501-525/',
        'hostid': '0501',
        'test': ["ecar/evaluation/24Sep19/AIA-501-525/AIA-501-525.ecar-2019-11-17T04-01-58.625.json.gz",
                "ecar/evaluation/24Sep19/AIA-501-525/AIA-501-525.ecar-last.json.gz"]
    },
    'optc_day25': {
        'path': 'ecar/benign/20-23Seq19/AIA-51-75/',
        'hostid': '0051',
        'test': ["ecar/evaluation/25Sept/AIA-51-75/AIA-51-75.ecar-last.json.gz"]
    }
}


for dataset in data_map:
    raw_data_path = f"{raw_optc_path}{data_map[dataset]['path']}"
    hostid = data_map[dataset]['hostid']
    output_directory = f"../host-detect/data/{dataset}/"
    # train data
    for file in os.listdir(raw_data_path):
        if '.json.gz' in file:
            index = file.find('.ecar')
            file_name = f"benign_20-23Seq19_{hostid}_{file[index:]}".removesuffix('.gz')
            output_path = output_directory + file_name
            extract_logs(raw_data_path + file, hostid, output_path)
    # test data
    test_file = data_map[dataset]['test']
    output_path = f"{output_directory}SysClient{hostid}.systemia.com.json"
    extract_logs(f"{raw_optc_path}{test_file[0]}", hostid, output_path)
    for file in test_file[1:]:
        extract_logs(f"{raw_optc_path}{file}", hostid, output_path, 'ab')


## 2. OpTC net data (zeek/bro)

* train: day 23; 0-12h
* validation: day 23; 12-14h
* day23 attack test data: day 23; 15-19h
* day24 attack test data: day 24; 14-21h
* day25 attack test data: day 25; 13-18h


host train: day20-20h to day23-10h

In [ ]:
import gzip
import os
import json


print(os.getcwd())
data_type = 'test'
day = '23'
data = f'2019-09-{day}/'
log_type = 'conn'
root_path = 'F:/datasets/OPTC/bro/' # replaced with your download path of bro/
time_list = [
            '00_00_00-01_00_00', '01_00_00-02_00_00', '02_00_00-03_00_00',
            '03_00_00-04_00_00', '04_00_00-05_00_00', '05_00_00-06_00_00',
            '06_00_00-07_00_00', '07_00_00-08_00_00', '08_00_00-09_00_00',
            '09_00_00-10_00_00', '10_00_00-11_00_00', '11_00_00-12_00_00',
            '12_00_00-13_00_00', '13_00_00-14_00_00', '14_00_00-15_00_00',
            '15_00_00-16_00_00', '16_00_00-17_00_00', '17_00_00-18_00_00',
            '18_00_00-19_00_00', '19_00_00-20_00_00', '20_00_00-21_00_00',
            '21_00_00-22_00_00', '22_00_00-23_00_00', '23_00_00-00_00_00'
            ]


start = 0
end = 10

gz_path = [f'{root_path}{data}{log_type}.{time}.log.gz' for time in time_list[start: end]]

for name in gz_path:
    print(name)

def merge_network(file_path, output_file):
    start_pattern = '#types'
    end_pattern = '#close'
    buffer = []
    with gzip.open(file_path, 'rt', encoding='utf-8') as fin:
        with open(output_file, 'a') as f:
            for line in fin:
                if start_pattern in line:
                    break
            for line in fin:
                if end_pattern in line:
                    break
                event = line_to_dict(line)
                buffer.append(json.dumps(event) + '\n')
                if len(buffer) >= 1000:
                    f.writelines(buffer)
                    buffer.clear()
            if buffer:
                f.writelines(buffer)


def line_to_dict(line):
    fields = line.split()
    event = {'timestamp': fields[0],
            'uid': fields[1],
            'src_ip_port': f'{fields[2]} {fields[3]}',
            'dest_ip_port': f'{fields[4]} {fields[5]}',
            'type': f'{fields[6]} {fields[7]}',
            }
    
    return event


output_file = f'./{data_type}_conn_{day}_{start}-{end}.json'
if os.path.exists(output_file):
    os.remove(output_file)
for file_path in gz_path:
    merge_network(file_path, output_file)

## 3. host-net-correlation (ecar-bro)

* The OpTC dataset contains correlation of host logs and net logs, which is consistent with the result obtained by our matching algorithm. Hence we directly use these files for convenience. 

In [ ]:
import gzip
import io


def extract_logs(filepath, hostid, output_filename):
    """unzip the datasets, filter out events of the host id

    filepath: the zip file path
    hostid: the host id of this data file
    """
    search_pattern = f'SysClient{hostid}'
    with gzip.open(filepath, 'rt', encoding='utf-8') as fin:
        with open(output_filename, 'wb') as f:
            out = io.BufferedWriter(f)
            for line in fin:
                if search_pattern in line:
                    out.write(line.encode('utf-8'))
            out.flush()

    print(f'extract {hostid} in {filepath} to {output_filename}')


# day23
zip_file = "ecar-bro/evaluation/23Sep19-red/AIA-201-225/ecarbro.json.gz"
file_path = '../net-detect/dataset/optc_day23-flow/ecarbro_23red_0201.json'
extract_logs(zip_file, '0201', file_path)

# day24
zip_file = r"F:\datasets\OPTC\ecar-bro\evaluation\24Sep19\AIA-501-525\ecarbro.json.gz"
file_path = '../net-detect/dataset/optc_day24-flow/ecarbro_24red_0501.json'
extract_logs(zip_file, '0501', file_path)

# day25
zip_file = "F:/datasets/OPTC/ecar-bro/evaluation/25Sept/AIA-51-75/ecarbro.json.gz"
file_path = '../net-detect/dataset/optc_day25-flow/ecarbro_25red_0051.json'
extract_logs(zip_file, '0051', file_path)